In [ ]:
# imports

import warnings
warnings.simplefilter('ignore', FutureWarning)
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import shapely
import igraph as ig
from pathlib import Path
from exactextract import exact_extract
import geohexgrid as ghg
from shapely.strtree import STRtree
import pyarrow 

import matplotlib.pyplot as plt
import contextily as cx
from matplotlib.colors import ListedColormap
from damagescanner.osm import _extract_value  # requires to pip install this package (rhttps://github.com/VU-IVM/DamageScanner). MUST BE LATEST DEV VERSION! (Pypi release is outdated.)

from snkit.network import Network, add_endpoints, split_edges_at_nodes, add_ids, add_topology

from simplify import * # requires to pip install osm-flex first. Python file simplify.py must be in the same folder as this notebook.
from pyproj import Geod

from functions import * 

warnings.simplefilter(action='ignore', category=FutureWarning)

c:\Users\gunaratn\AppData\Local\miniforge3\envs\MS2P_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Folder Paths

In [2]:
# Define root directory and network directory path
# the study area should be in the network directory, and the network directory should be in the root directory

root_dir = Path(r"P:\11212759-001-brs\002-Kaartverhal_KEA\MS2\MS2proposal\data\paramaribo\outputs_ver04")
network_path = root_dir/"data"/ "static" / "network"
output_path = root_dir/"data" /"static" / "output_graph"

workfolder_test = root_dir / "outputs" 

In [3]:
# areas
# area for population
HEX_AREA_FILE = network_path / "paramaribo_distrikt.gpkg" 

#area for roads and critical locations
Region = network_path / "paramaribo_distrikt_buffer.gpkg"

critical_locations_path = network_path / "osm_poi_selected.gpkg"

## Convert population tif into hexagons

In [11]:
# Downloaded population raster from WorldPop (https://www.worldpop.org/geodata/summary?id=24777) and clipped to the study area using the study area boundary shapefile
POP_FILE = network_path / "sur_pop_2026_CN_100m_R2025A_v1.tif"


# Hexagon size
HEX_RADIUS = 50  # meters
CRS_WGS84 = 4326
CRS_METRIC = 32621 

pop_ds = xr.open_dataset(POP_FILE)
pop_var = list(pop_ds.data_vars)[0]

population = pop_ds[pop_var].rio.write_crs(CRS_WGS84).rio.reproject(CRS_METRIC)
area = gpd.read_file(HEX_AREA_FILE).to_crs(CRS_METRIC)

hexgrid = ghg.make_grid_from_gdf(area, R=HEX_RADIUS)
pop_sum = exact_extract(population, hexgrid, 'sum', output='pandas')

hexgrid['population'] = pop_sum['sum'].values
hexgrid = hexgrid[hexgrid.population > 0].to_crs(CRS_WGS84)

origins = hexgrid.copy()
origins = origins.to_crs(CRS_METRIC)
origins['geometry'] = origins.centroid
origins = origins.to_crs(CRS_WGS84)
origins['type'] = 'origin'
origins = origins[['geometry','population','type']]

hexgrid.to_file(network_path / "hexgrid_dis.shp")

## Downloading networks

In [9]:
#Driving network configuration
from pathlib import Path
import geopandas as gpd
from shapely.geometry.polygon import Polygon

from ra2ce.network.network_config_data.enums.road_type_enum import RoadTypeEnum
from ra2ce.network.network_config_data.enums.network_type_enum import NetworkTypeEnum
from ra2ce.network.network_config_data.network_config_data import NetworkSection, NetworkConfigData, CleanupSection
from ra2ce.network.network_config_data.enums.source_enum import SourceEnum
from ra2ce.ra2ce_handler import Ra2ceHandler


gdf_polygon_path = network_path.joinpath(Region)
gdf_polygon = gpd.read_file(gdf_polygon_path)

network_section = NetworkSection(
    directed = True,
    source=SourceEnum.OSM_DOWNLOAD,
    network_type=NetworkTypeEnum.DRIVE,
    road_types=[RoadTypeEnum.MOTORWAY, RoadTypeEnum.MOTORWAY_LINK, 
                RoadTypeEnum.PRIMARY, RoadTypeEnum.PRIMARY_LINK,
                RoadTypeEnum.SECONDARY, RoadTypeEnum.SECONDARY_LINK,
                RoadTypeEnum.TERTIARY, RoadTypeEnum.TERTIARY_LINK,
                RoadTypeEnum.RESIDENTIAL, 
                RoadTypeEnum.UNCLASSIFIED,
                ],
    polygon=gdf_polygon_path,
    #attributes_to_exclude_in_simplification= ['bridge', 'tunnel'],
    reuse_network_output=True,
    save_gpkg=True,
)


network_config_data = NetworkConfigData(
    root_path=root_dir / "data",
    static_path=root_dir / "data"/"static",
    network=network_section,
    #cleanup = cleanup_section
)

handler = Ra2ceHandler.from_config(network=network_config_data, analysis=None)
handler.configure()

c:\Users\gunaratn\AppData\Local\miniforge3\envs\MS2P_env\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: driver GPKG does not support open option DRIVER
  return ogr_read(
c:\Users\gunaratn\AppData\Local\miniforge3\envs\MS2P_env\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: driver GPKG does not support open option DRIVER
  return ogr_read(


# Preprocessing for hazard scenarios

## Preprocessing the hazard map

In [12]:
hazard_map1 = root_dir / "data" / "static" / "hazard" / "Flood_rp_100.tif"

import rasterio
import numpy as np
from pathlib import Path

hazard_map = root_dir / "data" / "static" / "hazard" / "Flood_rp_100.tif"
output_raster = root_dir / "data" / "static" / "hazard" / "Flood_rp_100_nonnegative.tif"

with rasterio.open(hazard_map) as src:
    data = src.read(1)
    profile = src.profile.copy()

    nodata = profile.get("nodata", -9999)

    # Replace values below 0.1 with NoData
    data = np.where(data < 0.1, nodata, data)

    profile.update(nodata=nodata)

    with rasterio.open(output_raster, "w", **profile) as dst:
        dst.write(data, 1)

print(f"Saved to {output_raster}")

Saved to P:\11212759-001-brs\002-Kaartverhal_KEA\MS2\MS2proposal\data\paramaribo\outputs_ver04\data\static\hazard\Flood_rp_100_nonnegative.tif


In [8]:
hazard_map = root_dir / "data" / "static" / "hazard" / "Flood_rp_100_nonnegative.tif"

## Hazard overlay for roads

In [10]:
#Driving network configuration
from pathlib import Path
import geopandas as gpd
from ra2ce.network.network_config_data.enums.aggregate_wl_enum import AggregateWlEnum
from shapely.geometry.polygon import Polygon

from ra2ce.network.network_config_data.enums.road_type_enum import RoadTypeEnum
from ra2ce.network.network_config_data.enums.network_type_enum import NetworkTypeEnum
from ra2ce.network.network_config_data.network_config_data import NetworkSection, NetworkConfigData, HazardSection
from ra2ce.network.network_config_data.enums.source_enum import SourceEnum
from ra2ce.ra2ce_handler import Ra2ceHandler


gdf_polygon_path = network_path.joinpath(Region)
gdf_polygon = gpd.read_file(gdf_polygon_path)

network_section = NetworkSection(
    directed = True,
    source=SourceEnum.OSM_DOWNLOAD,
    network_type=NetworkTypeEnum.DRIVE,
    road_types=[RoadTypeEnum.MOTORWAY, RoadTypeEnum.MOTORWAY_LINK, 
                RoadTypeEnum.PRIMARY, RoadTypeEnum.PRIMARY_LINK,
                RoadTypeEnum.SECONDARY, RoadTypeEnum.SECONDARY_LINK,
                RoadTypeEnum.TERTIARY, RoadTypeEnum.TERTIARY_LINK,
                RoadTypeEnum.RESIDENTIAL, 
                RoadTypeEnum.UNCLASSIFIED,
                ],
    polygon=gdf_polygon_path,
    #attributes_to_exclude_in_simplification= ['bridge', 'tunnel'],
    reuse_network_output=True,
    save_gpkg=True,
)

hazard_section = HazardSection(
    hazard_map=[root_dir / "data" / "static" / "hazard" / "Flood_rp_100_nonnegative.tif"],
    aggregate_wl=AggregateWlEnum.MAX,
    hazard_crs="32621",
)

network_config_data = NetworkConfigData(
    root_path=root_dir / "data",
    static_path=root_dir / "data" / "static",
    network=network_section,
    hazard=hazard_section
    #cleanup = cleanup_section
)

handler = Ra2ceHandler.from_config(network=network_config_data, analysis=None)
handler.configure()

c:\Users\gunaratn\AppData\Local\miniforge3\envs\MS2P_env\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: driver GPKG does not support open option DRIVER
  return ogr_read(
c:\Users\gunaratn\AppData\Local\miniforge3\envs\MS2P_env\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: driver GPKG does not support open option DRIVER
  return ogr_read(


## Hazard overlay for population

In [14]:
# Hazard overlay for hexagons

import numpy as np
import geopandas as gpd
import rasterio
from rasterstats import gen_zonal_stats
from pathlib import Path
from tqdm.auto import tqdm

import warnings

warnings.simplefilter("ignore", UserWarning)

# -------------------------------------------------------------------
# Inputs
# -------------------------------------------------------------------
hexgrid = network_path / "hexgrid_dis.shp"
output_path = workfolder_test / "flooded_hexgrid_100rp_ver03.gpkg"
FLOOD_THRESHOLD_M = 0.1

# -------------------------------------------------------------------
# Load hexagon grid (polygons only)
# -------------------------------------------------------------------
gdf = gpd.read_file(hexgrid)

# Fix invalid geometries if needed
gdf["geometry"] = gdf.geometry.make_valid()

# Remove empty geometries
gdf = gdf[~gdf.geometry.is_empty & gdf.geometry.notna()].copy()


def _fraction_flooded(masked_array):
    # Fraction (0-1) of valid pixels in the hexagon with depth above threshold.
    valid = masked_array.compressed()
    if valid.size == 0:
        return np.nan
    return float((valid > FLOOD_THRESHOLD_M).sum()) / valid.size


def _mean_depth(masked_array):
    # Mean depth (m) over only flooded pixels above threshold.
    valid = masked_array.compressed()
    flooded = valid[valid > FLOOD_THRESHOLD_M]
    if flooded.size == 0:
        return np.nan
    return float(flooded.mean())


# -------------------------------------------------------------------
# Hazard overlay (hexagons are polygons, use max + custom stats per cell)
# -------------------------------------------------------------------
hazard_maps = [hazard_map] if isinstance(hazard_map, (str, Path)) else hazard_map
hazard_info = []

for raster_path in tqdm(hazard_maps, desc="Hazard rasters"):
    raster_path = Path(raster_path)

    hazard_name = "100R"
    value_col = f"{hazard_name}_value"
    exposed_col = f"{hazard_name}_exposed"
    fraction_col = f"{hazard_name}_fraction_flooded"
    mean_depth_col = f"{hazard_name}_mean_depth_m"

    with rasterio.open(raster_path) as src:
        raster_crs = src.crs
        nodata = src.nodata

        gdf_proj = gdf.to_crs(raster_crs) if gdf.crs != raster_crs else gdf.copy()

        polygon_stats = list(tqdm(
            gen_zonal_stats(
                gdf_proj.geometry,
                raster_path,
                stats=["max"],
                add_stats={"fraction_flooded": _fraction_flooded, "mean_depth": _mean_depth},
                nodata=nodata,
                all_touched=True,
            ),
            total=len(gdf_proj),
            desc=f"Zonal stats: {hazard_name}",
            leave=False,
        ))

    gdf[value_col] = [stat["max"] for stat in polygon_stats]
    gdf[value_col] = gdf[value_col].astype(float)
    gdf[exposed_col] = gdf[value_col].fillna(0) > FLOOD_THRESHOLD_M
    gdf[fraction_col] = [stat["fraction_flooded"] for stat in polygon_stats]
    gdf[mean_depth_col] = [stat["mean_depth"] for stat in polygon_stats]

    hazard_info.append((hazard_name, value_col, exposed_col, fraction_col, mean_depth_col))

# -------------------------------------------------------------------
# Summary
# -------------------------------------------------------------------
print(f"Total features: {len(gdf)}")
print(f"Flood threshold used: {FLOOD_THRESHOLD_M} m")

for hazard_name, value_col, exposed_col, fraction_col, mean_depth_col in hazard_info:
    print(f"\nHazard: {hazard_name}")
    print(f"Exposed features: {gdf[exposed_col].sum()}")
    print(f"Max value found: {gdf[value_col].max()}")
    print(f"Mean fraction flooded: {gdf[fraction_col].mean():.3f}")
    print(f"Mean depth (m) where flooded: {gdf[mean_depth_col].mean():.3f}")

# -------------------------------------------------------------------
# Save output
# -------------------------------------------------------------------
gdf.to_file(output_path, driver="GPKG")

print(f"\nSaved hazard overlay to: {output_path}")

Hazard rasters: 100%|██████████| 1/1 [02:11<00:00, 131.34s/it]


Total features: 20139
Flood threshold used: 0.1 m

Hazard: 100R
Exposed features: 10283
Max value found: 2.6621432304382324
Mean fraction flooded: 0.127
Mean depth (m) where flooded: 0.254

Saved hazard overlay to: P:\11212759-001-brs\002-Kaartverhal_KEA\MS2\MS2proposal\data\paramaribo\outputs_ver04\outputs\flooded_hexgrid_100rp_ver03.gpkg


## Hazard overlay for critical locations

In [15]:
# hazard overlay for critical locations

import geopandas as gpd
import rasterio
import numpy as np

locations = gpd.read_file(critical_locations_path)


locations = gpd.read_file("P:\\11212759-001-brs\\002-Kaartverhal_KEA\\MS2\\MS2proposal\\data\\paramaribo\\outputs_ver04\\outputs\\b_critical_locations.gpkg")
print(locations.columns)

with rasterio.open(hazard_map) as src:
    nodata = src.nodata

    # Sample the raster directly at each point so every location gets a value (or NaN)
    locations_proj = locations.to_crs(src.crs) if locations.crs != src.crs else locations

    # Use centroid so non-point geometries (e.g. polygons) can be sampled too
    coords = [(geom.x, geom.y) for geom in locations_proj.geometry.centroid]
    sampled_values = [val[0] for val in src.sample(coords)]

locations['flood_depth(m)'] = np.array(sampled_values, dtype=float)

# No-data / non-positive readings mean "not flooded", not a real depth
invalid_depth = (locations['flood_depth(m)'] <= 0)
if nodata is not None:
    invalid_depth |= (locations['flood_depth(m)'] == nodata)
locations.loc[invalid_depth, 'flood_depth(m)'] = np.nan

print(f"Points: {len(locations)}, with flood depth: {locations['flood_depth(m)'].notna().sum()}")

output_path = workfolder_test / "b_critical_locations_flooding.gpkg"
locations.to_file(output_path, driver="GPKG")
print(locations.columns)

Index(['population', 'type', 'category', 'Critical_Infra_id', 'geometry'], dtype='object')
Points: 377, with flood depth: 28
Index(['population', 'type', 'category', 'Critical_Infra_id', 'geometry',
       'flood_depth(m)'],
      dtype='object')
